### **Prerequisites for the Lab**

Before the session, ensure you have the environment set up.

1. Install [Ollama](https://ollama.com/) and download a lightweight model: `ollama run llama3` (or `llama3.1` or `mistral`).
2. Install the required Python libraries:

In [13]:
# Step 1: Install system compression tools and dependencies
!sudo apt-get update && sudo apt-get install -y zstd

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [14]:
# Step 2: Download and install the core Ollama environment binary
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [15]:
import os, subprocess, time

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'

# Open a log file to catch the output so it doesn't fill the OS pipe
log_file = open('/content/ollama_server.log', 'w')

subprocess.Popen(
    ["ollama", "serve"],
    stdout=log_file,
    stderr=log_file
)

time.sleep(5)

In [16]:
!ollama pull gemma4:latest

In [17]:
!pip install langchain langchain-ollama langchain-community

In [18]:
import os
import logging
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

### 1. Setup & Configuration

In [19]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [20]:
# Using the gemma model from your notebook setup
llm = ChatOllama(model="gemma4:latest", temperature=0.2)

### 2. Prompts (Enterprise Guidelines)

In [21]:
RECRUITER_SYSTEM_PROMPT = """You are a Senior Technical Recruiter at a Fortune 500 company.
Your task is to review the candidate's resume against the job description and draft a professional evaluation for the hiring manager, followed by a polite outreach email to the candidate.

Job Description: {job_description}
Candidate Resume: {resume}

If you receive feedback (critique) on your previous draft, you MUST incorporate that feedback and improve the draft.
Critique from TA Director: {critique}

Provide your response in two sections:
1. INTERNAL EVALUATION: (Your analysis of fit)
2. CANDIDATE EMAIL: (The email to send to the candidate)
"""

REFLECTOR_SYSTEM_PROMPT = """You are the Director of Talent Acquisition.
Your job is to review the Senior Recruiter's draft evaluation and outreach email.

Evaluate the draft against these enterprise criteria:
1. Tone: Is the email professional, welcoming, and inclusive?
2. Bias: Are there any biased assumptions made in the internal evaluation?
3. Accuracy: Does the evaluation accurately reflect the resume and job description?

Draft to review:
{current_draft}

If the draft meets all criteria perfectly, respond with exactly: "APPROVED".
If it needs improvement, provide a clear, constructive critique on what must be changed. Do not rewrite it yourself, just provide the critique.
"""

recruiter_prompt = ChatPromptTemplate.from_template(RECRUITER_SYSTEM_PROMPT)
reflector_prompt = ChatPromptTemplate.from_template(REFLECTOR_SYSTEM_PROMPT)

# Pre-compile the LangChain runnables
recruiter_chain = recruiter_prompt | llm
reflector_chain = reflector_prompt | llm

### 3. Main Agent Loop

In [22]:
def run_recruiter_workflow(job_description: str, resume: str, max_revisions: int = 3) -> str:
    """Executes the Recruiter-Reflector loop using native Python flow control."""

    # State is simply maintained in standard variables
    current_draft = ""
    critique = "No critique yet. This is the first draft."
    revision_count = 0

    logger.info("Starting Native Python Enterprise Recruiter Workflow...")

    # The 'while' loop replaces LangGraph's conditional routing
    while revision_count < max_revisions:
        logger.info(f"Generating draft (Revision {revision_count})...")

        # 1. Recruiter Agent Generation
        recruiter_response = recruiter_chain.invoke({
            "job_description": job_description,
            "resume": resume,
            "critique": critique
        })
        current_draft = recruiter_response.content

        print(f"\n--- RECRUITER DRAFT (Rev {revision_count}) ---")
        print(current_draft)

        # 2. Reflector Agent Critique
        logger.info("Director reviewing draft...")
        reflector_response = reflector_chain.invoke({
            "current_draft": current_draft
        })
        critique = reflector_response.content.strip()

        print("\n--- DIRECTOR CRITIQUE ---")
        print(critique)
        print("-" * 30)

        # 3. Routing Condition
        if "APPROVED" in critique.upper():
            logger.info("Draft APPROVED by Director.")
            break # Exit the loop immediately upon approval

        logger.info("Critique provided. Revisions required.")
        revision_count += 1

    # Failsafe check
    if revision_count >= max_revisions:
        logger.warning(f"Max revisions ({max_revisions}) reached. Terminating workflow to prevent infinite loops.")

    return current_draft

### 4. Execution Application

In [23]:
if __name__ == "__main__":
    sample_jd = """
    Role: Senior Python Developer
    Requirements: 5+ years of Python, experience with LangChain, REST APIs, and Docker.
    Must have excellent communication skills for cross-functional collaboration.
    """

    sample_resume = """
    Jane Doe
    Experience:
    - 6 years backend engineering using Python.
    - Built multiple GenAI apps using LangChain and OpenAI.
    - Familiar with Kubernetes and Docker containerization.
    """

    final_output = run_recruiter_workflow(sample_jd, sample_resume)
    print("\nFinal Result Ready.")


--- RECRUITER DRAFT (Rev 0) ---
***(Self-Correction/Review: The candidate's resume is very strong against the JD. I must ensure my internal evaluation doesn't sound too critical, but rather highly recommending, while still addressing the minor gap in explicit REST API mention.)***

---

## 1. INTERNAL EVALUATION (For Hiring Manager)

**Subject:** Candidate Review: Jane Doe - Senior Python Developer
**Recommendation:** Strong Hire / Proceed to Technical Interview Stage

Jane is a very promising candidate who appears to be an excellent fit for the technical requirements of the Senior Python Developer role, particularly given her recent experience in GenAI and LangChain. Her background aligns closely with our current project needs.

**Technical Fit Analysis (Green/Yellow/Red):**

*   **Python Experience (5+ years):** **🟢 Strong Match.** She reports 6 years of backend engineering using Python, exceeding the minimum requirement.
*   **LangChain / GenAI:** **🟢 Strong Match.** Explicitly bui